In [ ]:
import numpy as np
import pandas as pd
import json
import os
from datetime import datetime
from pandas._libs.tslibs.nattype import NaTType


EXCLUDE_FIELDS_IF_입고 = [
    "품목.number", "품목.수량", "품목.단가", "품목.공급가액", "품목.세액",
    "품목.수량합계", "품목.공급가액합계", "품목.세액합계"
]

def convert(v):
    if isinstance(v, (pd.Timestamp, datetime)):
        if v.hour == 0 and v.minute == 0 and v.second == 0:
            return v.date().isoformat()
        return v.isoformat()
    return v

def smart_number(value):
    try:
        if isinstance(value, str):
            value = value.strip().lstrip("'")
            if "," in value:
                result = []
                for x in value.split(","):
                    x = x.strip()
                    if x == "":
                        continue
                    f = float(x)
                    result.append(int(f) if f.is_integer() else f)
                return result
            if value == "":
                return ""
            f = float(value)
            return int(f) if f.is_integer() else f
        elif isinstance(value, (int, float)):
            if np.isnan(value) or np.isinf(value):
                return ""
            return int(value) if isinstance(value, float) and value.is_integer() else value
    except:
        return value

def is_invalid_excel_value(v):
    if isinstance(v, str) and v.startswith("#"):
        return True
    if isinstance(v, (pd.NA, pd.NaT, NaTType)):
        return True
    if isinstance(v, float) and (np.isnan(v) or np.isinf(v)):
        return True
    return False


# json_data 만들기 전에 리스트형 필드 강제 문자열 처리
def stringify_list_fields(d: dict, target_fields: list) -> dict:
    for k in target_fields:
        if k in d and isinstance(d[k], list):
            d[k] = ", ".join(str(v) for v in d[k])
    return d

def stringify_if_list(value):
    # 숫자 리스트 or 문자열 리스트 모두 → 문자열화
    if isinstance(value, list):
        return ", ".join(str(v) for v in value)
    return value
    

excel_path = r"data/labeling.xlsx"
output_dir = r"data/dataset"

numeric_fields = [
    "서류특성.합계금액", "피공급자.거래전미지급금", "피공급자.입금액", "피공급자.현잔액",
    "품목.number", "품목.수량", "품목.단가", "품목.공급가액", "품목.세액",
    "품목.수량합계", "품목.공급가액합계", "품목.세액합계"
]

os.makedirs(output_dir, exist_ok=True)

df = pd.read_excel(excel_path, header=0)
df[df.columns[0]] = df[df.columns[0]].astype(str)
df = df.rename(columns={df.columns[0]: "file_name"})
df = df.fillna("")

for field in numeric_fields:
    if field in df.columns:
        df[field] = df[field].apply(smart_number)

for idx, row in df.iterrows():
    raw_dict = row.to_dict()
    file_name_raw = raw_dict.pop("file_name")
    file_name = str(file_name_raw).strip()

    try:
        if file_name.replace(".", "", 1).isdigit() and float(file_name).is_integer():
            file_name = str(int(float(file_name)))
    except:
        pass

    if not file_name.lower().endswith(".pdf"):
        file_name += ".pdf"
    if not file_name or pd.isna(file_name):
        print(f"❗ 행 {idx+2}: file_name 누락 → 건너뜀")
        continue

    doc_type = str(raw_dict.get("서류특성.서류종류", "")).strip()
    json_data = {
        str(k): stringify_if_list(convert(v))  # ✅ 리스트 → 문자열
        for k, v in raw_dict.items()
        if str(v).strip() != "" and
           (doc_type != "입고서류" or str(k) not in EXCLUDE_FIELDS_IF_입고)

    }

    


    donut_json = {
        "file_name": file_name,
        "ground_truth": {
            "gt_parse": json_data
        }
    }

    json_filename = os.path.splitext(file_name)[0] + ".json"
    json_path = os.path.join(output_dir, json_filename)

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(donut_json, f, ensure_ascii=False, indent=4)

print(f"✅ 총 {len(df)}개의 Donut 학습용 JSON 파일 생성 완료! 저장 경로: {output_dir}")


In [ ]:
print(f"[DEBUG] {idx+1}행 file_name_raw = {repr(file_name_raw)} → file_name = {repr(file_name)}")

In [ ]:
print(df[["file_name"]].to_string(index=True))
